<a href="https://colab.research.google.com/github/HK25abm/Transformer-Phishing-Detection/blob/main/Random_Seed_Variation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

TRAIN_PATH = os.path.join(
    PROJECT_DIR,
    "train.csv"
)

VAL_PATH = os.path.join(
    PROJECT_DIR,
    "validation.csv"
)

TEST_PATH = os.path.join(
    PROJECT_DIR,
    "test.csv"
)

SEED_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "dissertation_results",
    "additional_analysis",
    "random_seeds"
)

os.makedirs(
    SEED_RESULTS_DIR,
    exist_ok=True
)

print("Output folder:", SEED_RESULTS_DIR)

Output folder: /content/drive/MyDrive/phishing_project/dissertation_results/additional_analysis/random_seeds


In [3]:
!pip -q install transformers datasets accelerate scikit-learn

In [4]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print(train_df.columns)

Train: (14000, 6)
Validation: (3000, 6)
Test: (3000, 6)
Index(['text', 'label', 'source', 'email_type', 'urls', 'sample_id'], dtype='object')


In [6]:
for df in [train_df, val_df, test_df]:
    df["text"] = (
        df["text"]
        .fillna("")
        .astype(str)
    )

    df["label"] = (
        df["label"]
        .astype(int)
    )

In [7]:
def set_seed_everywhere(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [8]:
MODEL_CONFIGS = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DeBERTa": "microsoft/deberta-v3-base"
}

In [9]:
def compute_metrics_from_predictions(
    y_true,
    y_pred,
    y_prob
):

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        )
    }

In [10]:
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

In [11]:
def run_seed_experiment(
    model_name,
    checkpoint,
    seed
):

    print("=" * 80)
    print(
        f"MODEL: {model_name} | SEED: {seed}"
    )
    print("=" * 80)

    set_seed_everywhere(seed)

    # -----------------------------
    # Tokenizer
    # -----------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        checkpoint
    )

    # -----------------------------
    # Hugging Face datasets
    # -----------------------------

    train_dataset = Dataset.from_pandas(
        train_df[
            ["text", "label"]
        ],
        preserve_index=False
    )

    val_dataset = Dataset.from_pandas(
        val_df[
            ["text", "label"]
        ],
        preserve_index=False
    )

    test_dataset = Dataset.from_pandas(
        test_df[
            ["text", "label"]
        ],
        preserve_index=False
    )

    def tokenize_function(batch):

        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=512
        )

    train_dataset = train_dataset.map(
        tokenize_function,
        batched=True
    )

    val_dataset = val_dataset.map(
        tokenize_function,
        batched=True
    )

    test_dataset = test_dataset.map(
        tokenize_function,
        batched=True
    )

    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer
    )

    # -----------------------------
    # Fresh model
    # -----------------------------

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=2
    )

    output_dir = os.path.join(
        SEED_RESULTS_DIR,
        f"{model_name}_seed_{seed}"
    )

    # -----------------------------
    # Training arguments
    # -----------------------------

    training_args = TrainingArguments(
        output_dir=output_dir,

        num_train_epochs=NUM_EPOCHS,

        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=
            TRAIN_BATCH_SIZE,

        per_device_eval_batch_size=
            EVAL_BATCH_SIZE,

        weight_decay=0.01,

        eval_strategy="epoch",

        save_strategy="no",

        logging_strategy="epoch",

        seed=seed,

        data_seed=seed,

        report_to="none",

        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,

        args=training_args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        tokenizer=tokenizer,

        data_collator=data_collator
    )

    # -----------------------------
    # Train
    # -----------------------------

    trainer.train()

    # -----------------------------
    # Test predictions
    # -----------------------------

    prediction_output = trainer.predict(
        test_dataset
    )

    logits = prediction_output.predictions

    probabilities = torch.softmax(
        torch.tensor(logits),
        dim=1
    ).numpy()

    y_prob = probabilities[:, 1]

    y_pred = np.argmax(
        probabilities,
        axis=1
    )

    y_true = test_df[
        "label"
    ].values

    metrics = (
        compute_metrics_from_predictions(
            y_true,
            y_pred,
            y_prob
        )
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred
    ).ravel()

    metrics.update({
        "model": model_name,
        "seed": seed,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    })

    # Save predictions
    prediction_df = pd.DataFrame({
        "true_label": y_true,
        "predicted_label": y_pred,
        "phishing_probability": y_prob
    })

    prediction_df.to_csv(
        os.path.join(
            output_dir,
            "test_predictions.csv"
        ),
        index=False
    )

    print(metrics)

    # Free GPU memory
    del trainer
    del model

    torch.cuda.empty_cache()

    return metrics

In [12]:
test_result = run_seed_experiment(
    model_name="BERT",
    checkpoint=MODEL_CONFIGS["BERT"],
    seed=42
)

test_result

MODEL: BERT | SEED: 42


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [13]:
import transformers
import datasets

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Transformers: 5.15.0
Datasets: 4.0.0


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

NameError: name 'model' is not defined

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

NameError: name 'model' is not defined

In [16]:
import os

PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

for root, dirs, files in os.walk(PROJECT_DIR):
    if "config.json" in files:
        print(root)

/content/drive/MyDrive/phishing_project/saved_models/bert
/content/drive/MyDrive/phishing_project/saved_models/roberta
/content/drive/MyDrive/phishing_project/saved_models/deberta
/content/drive/MyDrive/phishing_project/saved_models/deberta_fixed
/content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_adversarially_trained
/content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_trial2
/content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/roberta_final_defended
/content/drive/MyDrive/phishing_project/final_phase2_results/bert_final_defended
/content/drive/MyDrive/phishing_project/final_phase2_results/roberta_final_defended
/content/drive/MyDrive/phishing_project/final_phase2_results/deberta_final_defended


In [17]:
import os
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

TEST_PATH = os.path.join(
    PROJECT_DIR,
    "test.csv"
)

MODEL_PATHS = {
    "BERT":
        os.path.join(
            PROJECT_DIR,
            "saved_models",
            "bert"
        ),

    "RoBERTa":
        os.path.join(
            PROJECT_DIR,
            "saved_models",
            "roberta"
        ),

    "DeBERTa":
        os.path.join(
            PROJECT_DIR,
            "saved_models",
            "deberta_fixed"
        )
}

test_df = pd.read_csv(TEST_PATH)

test_df["text"] = (
    test_df["text"]
    .fillna("")
    .astype(str)
)

test_df["label"] = (
    test_df["label"]
    .astype(int)
)

print(test_df.shape)
print(test_df.columns.tolist())

(3000, 6)
['text', 'label', 'source', 'email_type', 'urls', 'sample_id']


In [18]:
def predict_with_saved_model(
    model_path,
    texts,
    batch_size=16,
    max_length=512
):

    tokenizer = AutoTokenizer.from_pretrained(
        model_path
    )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            model_path
        )
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model.to(device)
    model.eval()

    all_probs = []

    for i in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            i:i+batch_size
        ]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

        with torch.no_grad():

            outputs = model(
                **encoded
            )

            probs = torch.softmax(
                outputs.logits,
                dim=1
            )

        all_probs.append(
            probs.cpu().numpy()
        )

    return np.vstack(
        all_probs
    )

In [19]:
baseline_outputs = {}

texts = (
    test_df["text"]
    .tolist()
)

for model_name, model_path in (
    MODEL_PATHS.items()
):

    print(
        f"Running {model_name}..."
    )

    probs = predict_with_saved_model(
        model_path,
        texts
    )

    preds = np.argmax(
        probs,
        axis=1
    )

    model_df = test_df.copy()

    model_df[
        "predicted_label"
    ] = preds

    model_df[
        "legitimate_probability"
    ] = probs[:, 0]

    model_df[
        "phishing_probability"
    ] = probs[:, 1]

    baseline_outputs[
        model_name
    ] = model_df

    print(
        model_name,
        "done"
    )

Running BERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BERT done
Running RoBERTa...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa done
Running DeBERTa...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DeBERTa done


In [20]:
false_negative_tables = {}

for model_name, model_df in (
    baseline_outputs.items()
):

    fn_df = model_df[
        (
            model_df["label"] == 1
        )
        &
        (
            model_df[
                "predicted_label"
            ] == 0
        )
    ].copy()

    fn_df = fn_df.sort_values(
        "phishing_probability"
    )

    false_negative_tables[
        model_name
    ] = fn_df

    print(
        model_name,
        "false negatives:",
        len(fn_df)
    )

    display(
        fn_df[
            [
                "text",
                "label",
                "predicted_label",
                "phishing_probability"
            ]
        ].head(10)
    )

BERT false negatives: 20


,text,label,predicted_label,phishing_probability
981,hey ; ) if you need some extra umph in the bed...,1,0,0.000076
2522,"Free, no obligation quote: http://www.toplineq...",1,0,0.000089
1865,out of office autoreply : just to her . . . i ...,1,0,0.000137
1526,"offering closes march 31 , 1999 offering close...",1,0,0.000195
1807,Would you like to know what the Powerball Winn...,1,0,0.000296
677,Fw: Visit our New Discount Shop and KEEP your ...,1,0,0.000315
1030,BAD MSG:PAM: -------------------- Start SpamAs...,1,0,0.000366
2186,webmining free white paper on data mining web ...,1,0,0.000483
1220,"researcher. Well, three-a secretary.a lull.84 ...",1,0,0.000837
2587,Register and pay by May 31 and recieve 10% off...,1,0,0.001161


RoBERTa false negatives: 22


,text,label,predicted_label,phishing_probability
2587,Register and pay by May 31 and recieve 10% off...,1,0,0.000097
1865,out of office autoreply : just to her . . . i ...,1,0,0.000121
1807,Would you like to know what the Powerball Winn...,1,0,0.000125
118,"Dear Friend, Â ""Serving the Tribes While Shari...",1,0,0.000162
1118,would take hold. It was only a matter of time....,1,0,0.000309
1526,"offering closes march 31 , 1999 offering close...",1,0,0.000406
1201,RE: It-Support: Novo 2020 Microsoft Outlook Es...,1,0,0.000784
2828,Project Document Update Subcontractors/Supplie...,1,0,0.001000
1767,RE: [Highly Important] - Coronavirus outbreak ...,1,0,0.001009
2522,"Free, no obligation quote: http://www.toplineq...",1,0,0.001048


DeBERTa false negatives: 17


,text,label,predicted_label,phishing_probability
1767,RE: [Highly Important] - Coronavirus outbreak ...,1,0,0.000018
1763,"verna chang wed , 20 jul 2005 06 : 09 : 23 + 0...",1,0,0.000030
1865,out of office autoreply : just to her . . . i ...,1,0,0.000037
1143,Denise Austin's Morning Stretch Become a membe...,1,0,0.000056
239,Autodesk 3D Studio Max 2009 Creative Suite Mas...,1,0,0.000060
91,If you would like to see more information abou...,1,0,0.000069
118,"Dear Friend, Â ""Serving the Tribes While Shari...",1,0,0.000113
2587,Register and pay by May 31 and recieve 10% off...,1,0,0.000118
256,"re : hello 25 adefobi street , ikoyi island , ...",1,0,0.000188
353,"Deepest thanks to you, G & C, my numbers are g...",1,0,0.000355


In [21]:
ERROR_DIR = os.path.join(
    PROJECT_DIR,
    "dissertation_results",
    "additional_analysis",
    "false_negative_analysis"
)

os.makedirs(
    ERROR_DIR,
    exist_ok=True
)

for model_name, fn_df in (
    false_negative_tables.items()
):

    out_path = os.path.join(
        ERROR_DIR,
        f"{model_name.lower()}_false_negatives.csv"
    )

    fn_df.to_csv(
        out_path,
        index=False
    )

    print(
        "Saved:",
        out_path
    )

Saved: /content/drive/MyDrive/phishing_project/dissertation_results/additional_analysis/false_negative_analysis/bert_false_negatives.csv
Saved: /content/drive/MyDrive/phishing_project/dissertation_results/additional_analysis/false_negative_analysis/roberta_false_negatives.csv
Saved: /content/drive/MyDrive/phishing_project/dissertation_results/additional_analysis/false_negative_analysis/deberta_false_negatives.csv


In [22]:
CATEGORY_KEYWORDS = {

    "Lottery / Prize": [
        "winner",
        "powerball",
        "lottery",
        "jackpot",
        "claim prize"
    ],

    "Promotion / Advertisement": [
        "discount",
        "register",
        "offer",
        "free",
        "membership",
        "quote",
        "autodesk",
        "morning stretch",
        "promotion",
        "sale"
    ],

    "Business Communication": [
        "project",
        "support",
        "invoice",
        "document",
        "microsoft",
        "subcontractors",
        "update",
        "meeting",
        "contract"
    ],

    "Auto Reply": [
        "out of office",
        "autoreply",
        "automatic reply"
    ],

    "Health / COVID": [
        "coronavirus",
        "covid"
    ],

    "Charity / Donation": [
        "tribes",
        "charity",
        "donation"
    ],

    "Personal Conversation": [
        "hello",
        "thanks",
        "friend",
        "dear friend",
        "verna"
    ]
}

In [23]:
def classify_email(text):

    text = text.lower()

    for category, keywords in CATEGORY_KEYWORDS.items():

        for keyword in keywords:

            if keyword in text:

                return category

    return "Other"

In [24]:
category_summary = pd.DataFrame()

for model_name, df in false_negative_tables.items():

    counts = (
        df["category"]
        .value_counts()
        .rename(model_name)
    )

    category_summary = pd.concat(
        [category_summary, counts],
        axis=1
    )

category_summary = (
    category_summary
    .fillna(0)
    .astype(int)
)

display(category_summary)

KeyError: 'category'

In [25]:
for model_name, df in false_negative_tables.items():
    print(model_name, df.columns.tolist())

BERT ['text', 'label', 'source', 'email_type', 'urls', 'sample_id', 'predicted_label', 'legitimate_probability', 'phishing_probability']
RoBERTa ['text', 'label', 'source', 'email_type', 'urls', 'sample_id', 'predicted_label', 'legitimate_probability', 'phishing_probability']
DeBERTa ['text', 'label', 'source', 'email_type', 'urls', 'sample_id', 'predicted_label', 'legitimate_probability', 'phishing_probability']


In [26]:
CATEGORY_KEYWORDS = {

    "Lottery / Prize": [
        "winner",
        "powerball",
        "lottery",
        "jackpot",
        "claim prize"
    ],

    "Promotion / Advertisement": [
        "discount",
        "register",
        "offer",
        "free",
        "membership",
        "quote",
        "autodesk",
        "stretch",
        "promotion",
        "sale"
    ],

    "Business Communication": [
        "project",
        "support",
        "invoice",
        "document",
        "microsoft",
        "subcontractors",
        "meeting",
        "contract",
        "update"
    ],

    "Auto Reply": [
        "out of office",
        "autoreply"
    ],

    "Health / COVID": [
        "covid",
        "coronavirus"
    ],

    "Charity / Donation": [
        "tribes",
        "charity",
        "donation"
    ],

    "Personal Conversation": [
        "hello",
        "friend",
        "thanks",
        "dear friend",
        "verna"
    ]
}


def classify_email(text):

    text = str(text).lower()

    for category, keywords in CATEGORY_KEYWORDS.items():

        if any(keyword in text for keyword in keywords):
            return category

    return "Other"

In [27]:
for model_name in false_negative_tables.keys():

    false_negative_tables[model_name]["category"] = (

        false_negative_tables[model_name]["text"]

        .astype(str)

        .apply(classify_email)

    )

    print("\n", model_name)

    print(

        false_negative_tables[model_name]["category"]

        .value_counts()

    )


 BERT
category
Promotion / Advertisement    8
Other                        7
Lottery / Prize              2
Personal Conversation        2
Auto Reply                   1
Name: count, dtype: int64

 RoBERTa
category
Promotion / Advertisement    9
Other                        5
Business Communication       3
Lottery / Prize              2
Auto Reply                   1
Charity / Donation           1
Personal Conversation        1
Name: count, dtype: int64

 DeBERTa
category
Promotion / Advertisement    6
Other                        3
Personal Conversation        3
Business Communication       2
Auto Reply                   1
Lottery / Prize              1
Charity / Donation           1
Name: count, dtype: int64


In [28]:
for model_name in false_negative_tables:

    print(model_name)

    display(

        false_negative_tables[model_name][

            [
                "text",
                "category",
                "phishing_probability"
            ]

        ].head()

    )

BERT


,text,category,phishing_probability
981,hey ; ) if you need some extra umph in the bed...,Other,0.000076
2522,"Free, no obligation quote: http://www.toplineq...",Promotion / Advertisement,0.000089
1865,out of office autoreply : just to her . . . i ...,Auto Reply,0.000137
1526,"offering closes march 31 , 1999 offering close...",Promotion / Advertisement,0.000195
1807,Would you like to know what the Powerball Winn...,Lottery / Prize,0.000296


RoBERTa


,text,category,phishing_probability
2587,Register and pay by May 31 and recieve 10% off...,Promotion / Advertisement,0.000097
1865,out of office autoreply : just to her . . . i ...,Auto Reply,0.000121
1807,Would you like to know what the Powerball Winn...,Lottery / Prize,0.000125
118,"Dear Friend, Â ""Serving the Tribes While Shari...",Charity / Donation,0.000162
1118,would take hold. It was only a matter of time....,Other,0.000309


DeBERTa


,text,category,phishing_probability
1767,RE: [Highly Important] - Coronavirus outbreak ...,Business Communication,0.000018
1763,"verna chang wed , 20 jul 2005 06 : 09 : 23 + 0...",Personal Conversation,0.000030
1865,out of office autoreply : just to her . . . i ...,Auto Reply,0.000037
1143,Denise Austin's Morning Stretch Become a membe...,Lottery / Prize,0.000056
239,Autodesk 3D Studio Max 2009 Creative Suite Mas...,Promotion / Advertisement,0.000060


In [29]:
category_summary = pd.DataFrame()

for model_name, df in false_negative_tables.items():

    counts = (

        df["category"]

        .value_counts()

        .rename(model_name)

    )

    category_summary = pd.concat(

        [category_summary, counts],

        axis=1

    )

category_summary = (

    category_summary

    .fillna(0)

    .astype(int)

)

category_summary["Total"] = (

    category_summary.sum(axis=1)

)

category_summary = (

    category_summary

    .sort_values(

        "Total",

        ascending=False

    )

)

display(category_summary)

,BERT,RoBERTa,DeBERTa,Total
Promotion / Advertisement,8,9,6,23
Other,7,5,3,15
Personal Conversation,2,1,3,6
Lottery / Prize,2,2,1,5
Business Communication,0,3,2,5
Auto Reply,1,1,1,3
Charity / Donation,0,1,1,2


In [30]:
CATEGORY_TABLE_PATH = os.path.join(

    ERROR_DIR,

    "table_false_negative_categories.csv"

)

category_summary.to_csv(

    CATEGORY_TABLE_PATH,

    index=True

)

print("Saved:", CATEGORY_TABLE_PATH)

Saved: /content/drive/MyDrive/phishing_project/dissertation_results/additional_analysis/false_negative_analysis/table_false_negative_categories.csv


In [31]:
for model_name, df in false_negative_tables.items():

    print("\n" + "="*70)
    print(model_name)
    print("="*70)

    print("\nEMAIL TYPE")
    print(
        df["email_type"]
        .value_counts(dropna=False)
    )

    print("\nSOURCE")
    print(
        df["source"]
        .value_counts(dropna=False)
    )


BERT

EMAIL TYPE
email_type
phishing    20
Name: count, dtype: int64

SOURCE
source
Paper2    18
CEAS       2
Name: count, dtype: int64

RoBERTa

EMAIL TYPE
email_type
phishing    22
Name: count, dtype: int64

SOURCE
source
Paper2    13
CEAS       6
Modern     3
Name: count, dtype: int64

DeBERTa

EMAIL TYPE
email_type
phishing    17
Name: count, dtype: int64

SOURCE
source
Paper2    11
CEAS       4
Modern     2
Name: count, dtype: int64


In [32]:
email_type_summary = pd.DataFrame()

for model_name, df in false_negative_tables.items():

    counts = (
        df["email_type"]
        .fillna("Unknown")
        .value_counts()
        .rename(model_name)
    )

    email_type_summary = pd.concat(
        [email_type_summary, counts],
        axis=1
    )

email_type_summary = (
    email_type_summary
    .fillna(0)
    .astype(int)
)

email_type_summary["Total"] = (
    email_type_summary.sum(axis=1)
)

email_type_summary = (
    email_type_summary
    .sort_values("Total", ascending=False)
)

display(email_type_summary)

,BERT,RoBERTa,DeBERTa,Total
phishing,20,22,17,59


In [33]:
source_summary = pd.DataFrame()

for model_name, df in false_negative_tables.items():

    counts = (
        df["source"]
        .fillna("Unknown")
        .value_counts()
        .rename(model_name)
    )

    source_summary = pd.concat(
        [source_summary, counts],
        axis=1
    )

source_summary = (
    source_summary
    .fillna(0)
    .astype(int)
)

source_summary["Total"] = (
    source_summary.sum(axis=1)
)

source_summary = (
    source_summary
    .sort_values("Total", ascending=False)
)

display(source_summary)

,BERT,RoBERTa,DeBERTa,Total
Paper2,18,13,11,42
CEAS,2,6,4,12
Modern,0,3,2,5


In [34]:
email_type_summary.to_csv(
    os.path.join(
        ERROR_DIR,
        "table_false_negatives_by_email_type.csv"
    )
)

source_summary.to_csv(
    os.path.join(
        ERROR_DIR,
        "table_false_negatives_by_source.csv"
    )
)